In [1]:
import os, random, logging, sys
import numpy as np
import pandas as pd
from collections import Counter

import torch
from transformers import (
    AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, default_data_collator, set_seed
)
from datasets import Dataset, DatasetDict
import evaluate

In [ ]:
# Load the three prediction files
bert_predictions = "subtask_1C_bert.tsv"
indic_predictions = "subtask_1C_indic.tsv"
muril_predictions = "subtask_1C_muril.tsv"

# Read prediction files
df_bert = pd.read_csv(bert_predictions, sep="\t", keep_default_na=False)
df_indic = pd.read_csv(indic_predictions, sep="\t", keep_default_na=False)
df_muril = pd.read_csv(muril_predictions, sep="\t", keep_default_na=False)

print(f"BERT predictions shape: {df_bert.shape}")
print(f"IndicBERT predictions shape: {df_indic.shape}")
print(f"MuRIL predictions shape: {df_muril.shape}")

# Verify all files have same IDs and in same order
assert df_bert['id'].equals(df_indic['id']) and df_bert['id'].equals(df_muril['id']), "ID mismatch between files"
print("All prediction files have matching IDs")

# Get unique values for each task to create label mappings
hate_types = set(df_bert['hate_type'].unique()) | set(df_indic['hate_type'].unique()) | set(df_muril['hate_type'].unique())
hate_severities = set(df_bert['hate_severity'].unique()) | set(df_indic['hate_severity'].unique()) | set(df_muril['hate_severity'].unique())
to_whoms = set(df_bert['to_whom'].unique()) | set(df_indic['to_whom'].unique()) | set(df_muril['to_whom'].unique())

print(f"Hate types: {sorted(hate_types)}")
print(f"Hate severities: {sorted(hate_severities)}")
print(f"To whom: {sorted(to_whoms)}")

# Create label mappings for each task
hate_type_l2id = {label: i for i, label in enumerate(sorted(hate_types))}
hate_severity_l2id = {label: i for i, label in enumerate(sorted(hate_severities))}
to_whom_l2id = {label: i for i, label in enumerate(sorted(to_whoms))}

hate_type_id2l = {v: k for k, v in hate_type_l2id.items()}
hate_severity_id2l = {v: k for k, v in hate_severity_l2id.items()}
to_whom_id2l = {v: k for k, v in to_whom_l2id.items()}

BERT predictions shape: (2512, 5)
IndicBERT predictions shape: (2512, 5)
MuRIL predictions shape: (2512, 5)
✓ All prediction files have matching IDs
Hate types: ['Abusive', 'None', 'Political Hate', 'Profane', 'Religious Hate']
Hate severities: ['Little to None', 'Mild', 'Severe']
To whom: ['Community', 'Individual', 'None', 'Organization', 'Society']


In [3]:
# Convert predictions to integer IDs for each task
dfs = [df_bert, df_indic, df_muril]
model_names = ["bert", "indic", "muril"]

for df in dfs:
    df['hate_type_id'] = df['hate_type'].map(hate_type_l2id)
    df['hate_severity_id'] = df['hate_severity'].map(hate_severity_l2id)
    df['to_whom_id'] = df['to_whom'].map(to_whom_l2id)

# Get IDs for final output
ids = df_bert['id'].values

# Collect predictions for each task
hate_type_preds = [df['hate_type_id'].values for df in dfs]
hate_severity_preds = [df['hate_severity_id'].values for df in dfs]
to_whom_preds = [df['to_whom_id'].values for df in dfs]

print(f"Number of samples: {len(ids)}")
print(f"Number of models: {len(dfs)}")

# ===== HARD VOTING =====
print("\n=== Performing Hard Voting ===")

# Hard vote for hate_type
hard_hate_type = []
for i in range(len(ids)):
    votes = [preds[i] for preds in hate_type_preds]
    hard_hate_type.append(Counter(votes).most_common(1)[0][0])

# Hard vote for hate_severity
hard_hate_severity = []
for i in range(len(ids)):
    votes = [preds[i] for preds in hate_severity_preds]
    hard_hate_severity.append(Counter(votes).most_common(1)[0][0])

# Hard vote for to_whom
hard_to_whom = []
for i in range(len(ids)):
    votes = [preds[i] for preds in to_whom_preds]
    hard_to_whom.append(Counter(votes).most_common(1)[0][0])

# Convert back to labels
hard_hate_type_labels = [hate_type_id2l[pred] for pred in hard_hate_type]
hard_hate_severity_labels = [hate_severity_id2l[pred] for pred in hard_hate_severity]
hard_to_whom_labels = [to_whom_id2l[pred] for pred in hard_to_whom]

# Create hard voting submission
hard_submission = pd.DataFrame({
    "id": ids,
    "hate_type": hard_hate_type_labels,
    "hate_severity": hard_hate_severity_labels,
    "to_whom": hard_to_whom_labels,
    "model": "hv_ensemble_bert_indic_muril_1C"
})
hard_submission.to_csv("submission_hard_1C.tsv", sep="\t", index=False)
print("✓ Hard voting submission saved as 'submission_hard_1C.tsv'")

# ===== SOFT VOTING (Simple Average) =====
print("\n=== Performing Soft Voting ===")

# For soft voting, we'll use simple majority voting as we don't have probabilities
# This will be the same as hard voting for discrete predictions
soft_submission = hard_submission.copy()
soft_submission['model'] = "sv_ensemble_bert_indic_muril_1C"
soft_submission.to_csv("submission_soft_1C.tsv", sep="\t", index=False)
print("✓ Soft voting submission saved as 'submission_soft_1C.tsv' (same as hard voting for discrete predictions)")

# ===== WEIGHTED VOTING =====
print("\n=== Performing Weighted Voting ===")

# Define weights for each model (bert, indic, muril)
weights = [0.3, 0.2, 0.5]  # Adjust these based on model performance
print(f"Using weights: BERT={weights[0]}, IndicBERT={weights[1]}, MuRIL={weights[2]}")

# Weighted vote for hate_type
weighted_hate_type = []
for i in range(len(ids)):
    vote_counts = Counter()
    for j, preds in enumerate(hate_type_preds):
        vote_counts[preds[i]] += weights[j]
    weighted_hate_type.append(vote_counts.most_common(1)[0][0])

# Weighted vote for hate_severity
weighted_hate_severity = []
for i in range(len(ids)):
    vote_counts = Counter()
    for j, preds in enumerate(hate_severity_preds):
        vote_counts[preds[i]] += weights[j]
    weighted_hate_severity.append(vote_counts.most_common(1)[0][0])

# Weighted vote for to_whom
weighted_to_whom = []
for i in range(len(ids)):
    vote_counts = Counter()
    for j, preds in enumerate(to_whom_preds):
        vote_counts[preds[i]] += weights[j]
    weighted_to_whom.append(vote_counts.most_common(1)[0][0])

# Convert back to labels
weighted_hate_type_labels = [hate_type_id2l[pred] for pred in weighted_hate_type]
weighted_hate_severity_labels = [hate_severity_id2l[pred] for pred in weighted_hate_severity]
weighted_to_whom_labels = [to_whom_id2l[pred] for pred in weighted_to_whom]

# Create weighted voting submission
weighted_submission = pd.DataFrame({
    "id": ids,
    "hate_type": weighted_hate_type_labels,
    "hate_severity": weighted_hate_severity_labels,
    "to_whom": weighted_to_whom_labels,
    "model": "wv_ensemble_bert_indic_muril_1C"
})
weighted_submission.to_csv("submission_weighted_1C.tsv", sep="\t", index=False)
print("✓ Weighted voting submission saved as 'submission_weighted_1C.tsv'")

# ===== SUMMARY =====
print("\n=== Ensemble Summary ===")
print(f"Total samples processed: {len(ids)}")
print("\nHard Voting Results:")
print(f"  Hate Type distribution: {Counter(hard_hate_type_labels).most_common()}")
print(f"  Hate Severity distribution: {Counter(hard_hate_severity_labels).most_common()}")
print(f"  To Whom distribution: {Counter(hard_to_whom_labels).most_common()}")

print("\nWeighted Voting Results:")
print(f"  Hate Type distribution: {Counter(weighted_hate_type_labels).most_common()}")
print(f"  Hate Severity distribution: {Counter(weighted_hate_severity_labels).most_common()}")
print(f"  To Whom distribution: {Counter(weighted_to_whom_labels).most_common()}")

print("\n✓ All ensemble files created successfully!")

Number of samples: 2512
Number of models: 3

=== Performing Hard Voting ===
✓ Hard voting submission saved as 'submission_hard_1C.tsv'

=== Performing Soft Voting ===
✓ Soft voting submission saved as 'submission_soft_1C.tsv' (same as hard voting for discrete predictions)

=== Performing Weighted Voting ===
Using weights: BERT=0.3, IndicBERT=0.2, MuRIL=0.5
✓ Weighted voting submission saved as 'submission_weighted_1C.tsv'

=== Ensemble Summary ===
Total samples processed: 2512

Hard Voting Results:
  Hate Type distribution: [('None', 1571), ('Abusive', 441), ('Political Hate', 272), ('Profane', 192), ('Religious Hate', 36)]
  Hate Severity distribution: [('Little to None', 1876), ('Severe', 337), ('Mild', 299)]
  To Whom distribution: [('None', 1651), ('Individual', 321), ('Organization', 256), ('Society', 143), ('Community', 141)]

Weighted Voting Results:
  Hate Type distribution: [('None', 1571), ('Abusive', 382), ('Political Hate', 332), ('Profane', 190), ('Religious Hate', 37)]
  